# Report: Naive Bayes abstract classification
1. ### Data Representation and Preprocessing

The dataset consists of four classes: Archaea (A), Bacteria (B), Eukaryota (E), and Virus (V). The data is converted into a format suitable for Naive Bayes classification using a bag of words representation.

Preprocessing steps:

Each abstract is processed using the following transformations:

Conversion to lowercase, splitting on whitespace, Removal of non-informative stopwords such as “the” and “of”, Removing numeric strings

Merging of biologically meaningful multi-word expressions to avoid feature duplication, e.g:

“escherichia coli” -> “escherichia-coli”

“human immunodeficiency virus” -> “human-immunodeficiency-virus”

“homo sapian” -> “homo-sapian”

These reduce noise and prevent important biological terms from being split.

2. ### Feature Representation

A bag of words model is used, where every word is a feature and instead of storing raw text or a probability table, the model builds a frequency dictionary where:

Each word is a key in a dictionary, Each value is a list of counts corresponding to the four classes:

A, B, E, V

Two other data structures are maintained:

class_count: counts the number of documents per class

total_words_per_class: counts total word occurrences per class

This design allows efficient probability calculation without the need to precompute full probability tables.

3. ### Naive Bayes Model Implementation

The classifier is implemented from scratch using the Multinomial Naive Bayes formulation. To avoid numerical underflow, all probabilities are computed in log space.

The prior probability of each class is computed as:

P ( C ) = total number of documents / number of documents in class​

This captures the relative frequency of each biological domain in the dataset. 

Word likelihoods are estimated using Laplace smoothing:

P ( W ∣ C ) = total words in class + ∣ V ∣ / count( W, C ) + 1

where ∣ V ∣ is the vocabulary size.

This ensures that unseen words do not result in zero probability.

I played around with Laplace smoothing value to find optimal value

4. ### Prediction Strategy

For each abstract, the model:

Applies preprocessing to extract words, computes a score for each class, logs the prior probability, sums of the log likelihoods for all words and selects the class with the highest total log probability

5. ### Evaluation Procedure

Model performance is evaluated using classification accuracy:

Accuracy = total predictions / correct predictions

6. ### Results

Dataset	Accuracy:

Tr = training accuracy, Te = testing accuracy

Laplace value = 1.0: Tr = 97.95%, Te = 96.3%

Laplace value = 0.5: Tr = 99.38%, Te = 98.0%

Laplace value = 1.5: Tr = 96.45%, training accuracy too low

Laplace value = 0.3: Tr = 99.58%, Te = 96.6%

Laplace value = 0.7: Tr = 98.88%, Te = 97.0%

Observations:

The model achieves high accuracy on both training and test sets, suggesting good generalisation. The small gap between training and test accuracy indicates that the model is not significantly overfitting. Strong performance is largely due to domain-specific vocabulary in biological abstracts, which Naive Bayes handles effectively. Noticed that an increase in laplace value decreases training accuracy but increases testing accuracy. Found a laplace value of 0.5 is most efficient

7. ### Conclusion

My Naive Bayes classifier was successfully implemented from scratch for the classification. The model uses custom preprocessing, frequency-based feature representation, along with log-space probability computation with fine tuned Laplace smoothing.
The model achieves strong performance (98% test accuracy), demonstrating that Naive Bayes is highly effective for text classification tasks.

Reads the training and testing data so I am able to use them

In [1]:
import math
import pandas as pd

train_data = pd.read_csv("trg.csv")
test_data = pd.read_csv("tst.csv")

Preprocess any text put through the function and removes any common words like 'the' and 'of' to increase accuracy when predicting on the training and testing data, also combines two words together if shared with the same meaning as hinted at in canvas

In [2]:
def preprocess(text):
    remove = {"the", "of"}

    text = text.replace("homo sapian", "homo-sapian")
    text = text.replace("escherichia coli", "escherichia-coli")
    text = text.replace("human immunodeficiency virus", "human-immunodeficiency-virus")
    
    text = text.lower().split()
    text = [t for t in text if not t.isdigit()]
    result = [w for w in text if w not in remove]
    return result

This is my way of storing the data, instead of calculating probabilities fror each class for each word I have sorted the words into frequencys of each class and i've done this in a dictionary. This means I dont have a table of probabilities and instead have a dictionary where the key is the words and the value is a list of numbers. This does mean I have to calculate the probabilities later though, but this makes it a lot easier. Every time I run through one row I will add one to what ever class that instance was (class_count) and then add one for each word in the abstract of the row (total_words_per_class).

In [3]:
def get_word_count(text):
    word_count = {}
    class_count = {"A": 0, "B":0, "E":0, "V":0}
    total_words_per_class = {"A": 0, "B":0, "E":0, "V":0}
    for i in range(len(text)):

        if text["class"][i] == "A":
            words = preprocess(text["abstract"][i])
            for word in words:
                if word not in word_count:
                    word_count[word] = [1, 0, 0, 0]
                else:
                    word_count[word][0] += 1
                total_words_per_class["A"] += 1
            class_count["A"] += 1
            

        elif text["class"][i] == "B":
            words = preprocess(text["abstract"][i])
            for word in words:
                if word not in word_count:
                    word_count[word] = [0, 1, 0, 0]
                else:
                    word_count[word][1] += 1
                total_words_per_class["B"] += 1
            class_count["B"] += 1

        elif text["class"][i] == "E":
            words = preprocess(text["abstract"][i])
            for word in words:
                if word not in word_count:
                    word_count[word] = [0, 0, 1, 0]
                else:
                    word_count[word][2] += 1
                total_words_per_class["E"] += 1
            class_count["E"] += 1

        elif text["class"][i] == "V":
            words = preprocess(text["abstract"][i])
            for word in words:
                if word not in word_count:
                    word_count[word] = [0, 0, 0, 1]
                else:
                    word_count[word][3] += 1
                total_words_per_class["V"] += 1
            class_count["V"] += 1

    return word_count, class_count, total_words_per_class

Logging helpers help me log each individual score for log_likelihood so I can then later add up the total in my predict function instead of multiplying all the values and then logging which fixes computaional issues as stated in canvas. log_prior calculates how common the class is or the probability of each class showing up. This should only run 4 times.

In [4]:
def log_prior(class_count, class_label):
   total = sum(class_count.values())
   return math.log(class_count[class_label] / total)

def log_likelihood(word, class_label, word_count, total_words_per_class):
   class_index = {
      "A": 0,
      "B": 1,
      "E": 2,
      "V": 3
   }
   i = class_index[class_label]

   count = word_count.get(word, [0, 0, 0, 0])[i]

   return math.log((count + 0.7) / (total_words_per_class[class_label] + 0.7*len(word_count)))

This function predicts the class of a given abstract using the model I made from get_word_count. it preprocesses the text into words then for every class label it caltulates a score made up of log_prior probability plus the sum of the log_likelihoods of each word in the class. and then getting the highest score of the four classes and returns that label.

In [5]:
def predict(text, word_count, class_count, total_words_per_class):
    words = preprocess(text)
    classes = ["A", "B", "E", "V"]
    best_class = None
    best_score = float("-inf")

    for class_label in classes:
        score = log_prior(class_count, class_label)
        for word in words:
           score += log_likelihood(word, class_label, word_count, total_words_per_class) 

        if score > best_score:
            best_score = score
            best_class = class_label

    return best_class

Function to evalute how accurate the model is using the model we have trained against the data we want to test whether it is for training or testing accuracy

In [6]:
def evaluate(test_data, word_count, class_count, total_words_per_class):
    correct = 0

    for i in range(len(test_data)):
        prediction = predict(test_data["abstract"][i], word_count, class_count, total_words_per_class)
        if prediction == test_data["class"][i]:
            correct += 1

    return round(correct / len(test_data), 4)

Evaluating my model trained on the training data, and then testing it against the training data

In [7]:
word_count, class_count, total_words_per_class = get_word_count(train_data)

print("Training Accuracy:", str(evaluate(train_data, word_count, class_count, total_words_per_class)*100) + "%")


Training Accuracy: 98.88%


Finally training my model on the training data and then making predictions on the testing data using my model.

In [8]:
word_count, class_count, total_words_per_class = get_word_count(train_data)

predictions = []

for i in range(len(test_data)):
    text = test_data["abstract"].iloc[i]
    pred = predict(text, word_count, class_count, total_words_per_class)
    predictions.append(pred)

submission = pd.DataFrame({"id": test_data["id"],"class": predictions})

submission.to_csv("submission.csv", index=False)

I got a 98% on the testing data predictions when I submitted to kaggle which I think is really good, and shows that my model wasnt overfit. This means that my model is able to accuratly geuss what field any abstract text is in.